In [ ]:
!pip install -q psycopg2-binary pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 23.4 MB/s eta 0:00:00


In [ ]:
from google.colab import files

uploaded = files.upload()

SOURCE_FILE = next(iter(uploaded))
print(f"Using file: {SOURCE_FILE}")

Saving timeslots_input_synthetic.xlsx to timeslots_input_synthetic.xlsx
Using file: timeslots_input_synthetic.xlsx


In [ ]:
# ============================================================
# ESK_CORE_V3 LOADER — COLAB VERSION
# ============================================================

import logging
import uuid
import re

import pandas as pd
import psycopg2
import psycopg2.extras


# ============================================================
# SETUP
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

log = logging.getLogger("esk_core_v3_loader")

NS = uuid.UUID(
    "01a09b7a-0000-7000-8000-000000000000"
)

ROLE_TEXT = {
    "Mentor": "mentor",
    "Instructor": "instructor",
    "Operator": "operator"
}

ESKWELABS_PARTY_ID = (
    "019d3d9a-0000-7000-8000-000000000020"
)

FIXED_VALID_FROM = "2026-09-15 00:00:00+00"


# ============================================================
# HELPER
# ============================================================

def slug(s):
    return (
        re.sub(
            r"[^a-z0-9_]",
            "_",
            str(s).lower()
        )
        .strip("_")
    )


# ============================================================
# TRANSFORM
# ============================================================

def build_rows(source_path):

    log.info(
        f"Reading Excel file: {source_path}"
    )

    df = pd.read_excel(
        source_path,
        sheet_name="Sheet1"
    )

    # --------------------------------------------------------
    # CLEAN SOURCE DATA
    # --------------------------------------------------------

    df = df.dropna(
        how="all",
        subset=list(df.columns)
    )

    df = df[
        df["timeslot_id"].notna()
    ].reset_index(drop=True)

    log.info(
        f"Source rows after cleaning: {len(df)}"
    )

    # ========================================================
    # PARTY
    # ========================================================

    party_rows = []

    staff_ids = pd.unique(
        pd.concat([
            df["assigned_party_id"],
            df["pm_party_id"]
        ]).dropna()
    )

    for pid in staff_ids:

        party_rows.append((
            pid,
            "person",
            f"Staff {str(pid)[-6:]}",
            None,
            "active"
        ))

    # Add Eskwelabs organization
    party_rows.append((
        ESKWELABS_PARTY_ID,
        "organization",
        "Eskwelabs",
        "formal",
        "active"
    ))

    # ========================================================
    # PROJECT
    # ========================================================

    proj = (
        df.groupby(
            [
                "project_id",
                "Project_Name",
                "pm_party_id"
            ],
            dropna=False
        )
        .size()
        .reset_index()
    )

    proj = proj[
        proj["project_id"].notna()
    ]

    project_rows = []

    for _, r in proj.iterrows():

        # Convert pandas NaN to Python None
        # so PostgreSQL receives NULL instead
        # of a double-precision NaN value.
        owner_party_id = (
            r["pm_party_id"]
            if pd.notna(r["pm_party_id"])
            else None
        )

        project_rows.append((
            r["project_id"],
            slug(r["Project_Name"]),
            r["Project_Name"],
            "learning_experience",
            owner_party_id,
            "active"
        ))

    # ========================================================
    # PARTY ROLE
    # ========================================================

    roles = (
        df[
            [
                "assigned_party_id",
                "Role",
                "project_id"
            ]
        ]
        .dropna()
        .drop_duplicates()
    )

    party_role_rows = []

    for _, r in roles.iterrows():

        role_type = ROLE_TEXT.get(
            r["Role"],
            slug(r["Role"])
        )

        party_role_rows.append((
            r["assigned_party_id"],
            role_type,
            r["project_id"],
            FIXED_VALID_FROM,
            "active"
        ))

    # ========================================================
    # OBLIGATION SET
    # ========================================================

    set_ids = {}

    obligation_set_rows = []

    for _, r in proj.iterrows():

        set_key = (
            f"staffing_{slug(r['Project_Name'])}"
        )

        set_id = str(
            uuid.uuid5(
                NS,
                set_key
            )
        )

        set_ids[r["project_id"]] = set_id

        obligation_set_rows.append((
            set_id,
            r["project_id"],
            set_key,
            "Staffing - " + str(r["Project_Name"]),
            "staffing",
            "active"
        ))

    # ========================================================
    # OBLIGATION / OCCURRENCE
    # ========================================================

    obligation_rows = []
    occurrence_rows = []
    obligation_occurrence_rows = []

    for _, r in df.iterrows():

        # Skip incomplete records
        if (
            pd.isna(r["project_id"])
            or pd.isna(r["assigned_party_id"])
            or pd.isna(r["Role"])
            or pd.isna(r["Hours"])
        ):
            continue

        set_id = set_ids.get(
            r["project_id"]
        )

        if set_id is None:
            continue

        rtxt = ROLE_TEXT.get(
            r["Role"],
            slug(r["Role"])
        )

        subtype = (
            f"{rtxt}_session_delivery"
        )

        # Convert blank dates to None
        due = (
            r["Date"]
            if pd.notna(r["Date"])
            else None
        )

        # Convert blank PM IDs to Eskwelabs
        beneficiary = (
            r["pm_party_id"]
            if pd.notna(r["pm_party_id"])
            else ESKWELABS_PARTY_ID
        )

        ob_key = (
            f"session_{r['timeslot_id']}"
        )

        # ----------------------------------------------------
        # OBLIGATION
        # ----------------------------------------------------

        obligation_rows.append((
            r["timeslot_id"],
            set_id,
            ob_key,
            "service_performance",
            subtype,
            "general_requirement",
            r["assigned_party_id"],
            beneficiary,
            (
                f"{r['Role']} to deliver "
                f"{r['Hours']}h for "
                f"{r['Project_Name']}"
            ),
            float(r["Hours"]),
            "hour",
            due,
            "active",
            "resolved"
        ))

        # ----------------------------------------------------
        # OCCURRENCE
        # ----------------------------------------------------

        if r["status"] == "present":

            occ_id = str(
                uuid.uuid5(
                    NS,
                    f"occ-{r['timeslot_id']}"
                )
            )

            occurrence_rows.append((
                occ_id,
                r["project_id"],
                "service_performance",
                subtype,
                due,
                r["assigned_party_id"],
                float(r["Hours"]),
                "hour",
                "synthetic_timeslots_import",
                r["timeslot_id"]
            ))

            obligation_occurrence_rows.append((
                r["timeslot_id"],
                occ_id,
                float(r["Hours"]),
                f"alloc_{r['timeslot_id']}"
            ))

    # ========================================================
    # RETURN ALL TRANSFORMED DATA
    # ========================================================

    return {
        "party": party_rows,
        "project": project_rows,
        "party_role": party_role_rows,
        "obligation_set": obligation_set_rows,
        "obligation": obligation_rows,
        "occurrence": occurrence_rows,
        "obligation_occurrence": obligation_occurrence_rows,
    }


# ============================================================
# DRY RUN / ROW COUNT CHECK
# ============================================================

def show_dry_run(rows):

    print("\n" + "=" * 60)
    print("DRY RUN — ROW COUNTS")
    print("=" * 60)

    total = 0

    for table, data in rows.items():

        count = len(data)
        total += count

        print(
            f"{table:<25} {count:>8,} rows"
        )

    print("-" * 60)

    print(
        f"{'TOTAL':<25} {total:>8,} rows"
    )

    print("=" * 60)
    print("No database changes have been made.")
    print("=" * 60)


# ============================================================
# DATABASE LOAD
# ============================================================

def load(rows, dsn):

    log.info(
        "Connecting to Supabase PostgreSQL..."
    )

    conn = psycopg2.connect(dsn)

    conn.autocommit = False

    try:

        with conn.cursor() as cur:

            # ------------------------------------------------
            # SET SCHEMA
            # ------------------------------------------------

            cur.execute(
                "SET search_path = esk_core_v3, pg_catalog"
            )

            # ------------------------------------------------
            # PARTY
            # ------------------------------------------------

            log.info(
                "Loading party (%d rows)...",
                len(rows["party"])
            )

            psycopg2.extras.execute_values(
                cur,
                """
                INSERT INTO party (
                    party_id,
                    party_kind,
                    display_name,
                    organization_form,
                    status
                )
                VALUES %s
                ON CONFLICT (party_id)
                DO NOTHING
                """,
                rows["party"]
            )

            # ------------------------------------------------
            # PROJECT
            # ------------------------------------------------

            log.info(
                "Loading project (%d rows)...",
                len(rows["project"])
            )

            psycopg2.extras.execute_values(
                cur,
                """
                INSERT INTO project (
                    project_id,
                    project_key,
                    project_name,
                    project_type,
                    owner_party_id,
                    status
                )
                VALUES %s
                ON CONFLICT (project_id)
                DO NOTHING
                """,
                rows["project"]
            )

            # ------------------------------------------------
            # PARTY ROLE
            # ------------------------------------------------

            log.info(
                "Loading party_role (%d rows)...",
                len(rows["party_role"])
            )

            psycopg2.extras.execute_values(
                cur,
                """
                INSERT INTO party_role (
                    party_id,
                    role_type,
                    scope_project_id,
                    valid_from,
                    status
                )
                VALUES %s
                ON CONFLICT DO NOTHING
                """,
                rows["party_role"]
            )

            # ------------------------------------------------
            # OBLIGATION SET
            # ------------------------------------------------

            log.info(
                "Loading obligation_set (%d rows)...",
                len(rows["obligation_set"])
            )

            psycopg2.extras.execute_values(
                cur,
                """
                INSERT INTO obligation_set (
                    obligation_set_id,
                    project_id,
                    set_key,
                    set_name,
                    set_type,
                    status
                )
                VALUES %s
                ON CONFLICT (set_key)
                DO NOTHING
                """,
                rows["obligation_set"]
            )

            # ------------------------------------------------
            # OBLIGATION
            # ------------------------------------------------

            log.info(
                "Loading obligation (%d rows)...",
                len(rows["obligation"])
            )

            psycopg2.extras.execute_values(
                cur,
                """
                INSERT INTO obligation (
                    obligation_id,
                    obligation_set_id,
                    obligation_key,
                    outcome,
                    subtype,
                    authority,
                    obligor_party_id,
                    beneficiary_party_id,
                    description,
                    required_quantity,
                    unit_code,
                    due_at,
                    lifecycle_status,
                    source_resolution_status
                )
                VALUES %s
                ON CONFLICT (obligation_id)
                DO NOTHING
                """,
                rows["obligation"]
            )

            # ------------------------------------------------
            # OCCURRENCE
            # ------------------------------------------------

            log.info(
                "Loading occurrence (%d rows)...",
                len(rows["occurrence"])
            )

            psycopg2.extras.execute_values(
                cur,
                """
                INSERT INTO occurrence (
                    occurrence_id,
                    project_id,
                    outcome,
                    subtype,
                    occurred_at,
                    actor_party_id,
                    quantity,
                    unit_code,
                    source_system,
                    source_record_id
                )
                VALUES %s
                ON CONFLICT (
                    source_system,
                    source_record_id,
                    source_event_key
                )
                DO NOTHING
                """,
                rows["occurrence"]
            )

            # ------------------------------------------------
            # OBLIGATION OCCURRENCE
            # ------------------------------------------------

            log.info(
                "Loading obligation_occurrence (%d rows)...",
                len(rows["obligation_occurrence"])
            )

            psycopg2.extras.execute_values(
                cur,
                """
                INSERT INTO obligation_occurrence (
                    obligation_id,
                    occurrence_id,
                    allocated_quantity,
                    allocation_key
                )
                VALUES %s
                ON CONFLICT (allocation_key)
                DO NOTHING
                """,
                rows["obligation_occurrence"]
            )

        # ----------------------------------------------------
        # COMMIT
        # ----------------------------------------------------

        conn.commit()

        log.info(
            "All tables loaded and committed successfully."
        )

        print("\n" + "=" * 60)
        print("✅ SUCCESS")
        print("=" * 60)
        print(
            "Data successfully loaded into esk_core_v3."
        )
        print("=" * 60)

    except Exception:

        conn.rollback()

        log.exception(
            "Load failed — transaction rolled back."
        )

        print("\n❌ LOAD FAILED")
        print(
            "The transaction was rolled back."
        )
        print(
            "No partial changes were committed."
        )

        raise

    finally:

        conn.close()

        log.info(
            "Database connection closed."
        )


# ============================================================
# BUILD TRANSFORMED ROWS
# ============================================================

rows = build_rows(SOURCE_FILE)

# ============================================================
# SHOW DRY RUN
# ============================================================

show_dry_run(rows)


DRY RUN — ROW COUNTS
party                           62 rows
project                         83 rows
party_role                     230 rows
obligation_set                  83 rows
obligation                   1,106 rows
occurrence                     983 rows
obligation_occurrence          983 rows
------------------------------------------------------------
TOTAL                        3,530 rows
No database changes have been made.


In [ ]:
# ============================================================
# SUPABASE CONNECTION
# ============================================================

print("\nEnter the Supabase PostgreSQL connection string.")
print("The value will NOT be displayed while typing.\n")

dsn = getpass.getpass("Supabase DSN: ")

if not dsn.strip():
    raise ValueError("No DSN provided.")

print("\nConnection string received.")


Enter the Supabase PostgreSQL connection string.
The value will NOT be displayed while typing.

Supabase DSN: ··········

Connection string received.


In [ ]:
load(rows, dsn)


✅ SUCCESS
Data successfully loaded into esk_core_v3.
